<div style="display:flex; justify-content:space-between; align-items:center; width:100%; margin:8px 0 24px 0;">
  <div style="text-align:left;">
    <img src="https://www.ec-nantes.fr/medias/photo/logocn-rvb_1648479844750-png?ID_FICHE=178994&amp;INLINE=FALSE" alt="Centrale Nantes" style="height:72px; width:auto;">
  </div>
  <div style="text-align:right; font-size:18px; font-weight:600; color:#17324d; line-height:1.35;">
    MSc. CORO DASSIP
  </div>
</div>

<div style="border:2px solid #333; padding:14px 20px; margin:15px auto 25px auto; width:85%; max-width:900px; box-sizing:border-box; text-align:center;">
  <h1 style="margin:0;"><b>Filtering in Spatial Domain — Implementation</b></h1>
</div>

## Setup — Environment and Configuration


In [ ]:
from pathlib import Path
import math

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage
import cv2

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 11

## 1. Data and Output Paths


In [ ]:
def locate_lab_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate the lab root containing data/ and notebooks/."
    )
LAB_DIR = locate_lab_root(Path.cwd())
DATA_DIR = LAB_DIR / "data"
OUTPUT_DIR = LAB_DIR / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
DATA_FILES = sorted(
    path for path in DATA_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

assert DATA_FILES, f"No supported images found in: {DATA_DIR}"
def select_input_image(*tokens: str) -> Path:
    tokens = tuple(token.lower() for token in tokens)
    matches = [
        path for path in DATA_FILES
        if all(
            token in path.relative_to(DATA_DIR).as_posix().lower()
            for token in tokens
        )
    ]
    if not matches:
        raise FileNotFoundError(
            f"No image matching {tokens} found in {DATA_DIR}"
        )
    return matches[0]
ROLE_PATTERNS = {
    "moon": ("moon-blurred",),
    "ascent": ("ascentb",),
    "einstein_ref": ("einstein/ref",),
    "einstein_gaussian": ("einstein_gaussian",),
    "einstein_salt": ("saltandpepper",),
    "einstein_speckle": ("speckle",),
    "taj_ref": ("tajmahal/ref",),
    "zebra_ref": ("zebra/ref",),
    "monarch_ref": ("monarch/ref",),
}

INPUT_IMAGES = {
    role: select_input_image(*patterns)
    for role, patterns in ROLE_PATTERNS.items()
}

print("Lab directory :", LAB_DIR)
print("Data directory:", DATA_DIR)
print("Images discovered:", len(DATA_FILES))
print("Experiment roles :", len(INPUT_IMAGES))
print("Output folder :", OUTPUT_DIR)

### Data Inventory


In [ ]:
def load_grayscale_image(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"), dtype=np.uint8)
def load_rgb_image(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"), dtype=np.uint8)
einstein_reference_image = load_grayscale_image(INPUT_IMAGES["einstein_ref"])
einstein_gaussian_noise_image = load_grayscale_image(INPUT_IMAGES["einstein_gaussian"])
einstein_salt_pepper_noise_image = load_grayscale_image(INPUT_IMAGES["einstein_salt"])
einstein_speckle_noise_image = load_grayscale_image(INPUT_IMAGES["einstein_speckle"])

moon = load_grayscale_image(INPUT_IMAGES["moon"])
ascent = load_grayscale_image(INPUT_IMAGES["ascent"])

taj_reference_image = load_rgb_image(INPUT_IMAGES["taj_ref"])
zebra_reference_image = load_rgb_image(INPUT_IMAGES["zebra_ref"])
monarch_reference_image = load_rgb_image(INPUT_IMAGES["monarch_ref"])

for name, image in {
    "einstein_ref": einstein_reference_image,
    "einstein_gaussian": einstein_gaussian_noise_image,
    "einstein_salt": einstein_salt_pepper_noise_image,
    "einstein_speckle": einstein_speckle_noise_image,
    "moon": moon,
    "ascent": ascent,
    "taj_ref": taj_reference_image,
    "zebra_ref": zebra_reference_image,
    "monarch_ref": monarch_reference_image,
}.items():
    print(
        f"{name:20s} shape={image.shape!s:16s} "
        f"dtype={image.dtype} range=({image.min()}, {image.max()})"
    )

## 2. Spatial Filtering Formulation


In [ ]:
toy = np.array(
    [
        [10, 10, 10, 10, 10],
        [10, 20, 20, 20, 10],
        [10, 20, 90, 20, 10],
        [10, 20, 20, 20, 10],
        [10, 10, 10, 10, 10],
    ],
    dtype=np.float32,
)

center_neighborhood = toy[1:4, 1:4]

print("Toy image:")
print(toy)
print("\n3x3 neighborhood around the center:")
print(center_neighborhood)

## 3. Kernel Properties and Normalization


In [ ]:
mean_3x3 = np.ones((3, 3), dtype=np.float32) / 9.0
sobel_x_kernel = np.array(
    [
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1],
    ],
    dtype=np.float32,
)

print("Mean kernel:")
print(mean_3x3)
print("Sum:", mean_3x3.sum())

print("\nSobel-x kernel:")
print(sobel_x_kernel)
print("Sum:", sobel_x_kernel.sum())

## 4. Correlation vs Convolution


In [ ]:
small = np.arange(1, 26, dtype=np.float32).reshape(5, 5)
asymmetric_kernel = np.array(
    [
        [1, 2, 0],
        [0, 0, 0],
        [0, 0, -1],
    ],
    dtype=np.float32,
)
flipped_kernel = np.flip(asymmetric_kernel, axis=(0, 1))

print("Input:")
print(small)
print("\nOriginal kernel:")
print(asymmetric_kernel)
print("\nKernel flipped for convolution:")
print(flipped_kernel)

In [ ]:
patch = small[1:4, 1:4]

manual_corr = np.sum(patch * asymmetric_kernel)
manual_conv = np.sum(patch * flipped_kernel)

print("Patch:")
print(patch)
print("\nManual correlation response:", manual_corr)
print("Manual convolution response :", manual_conv)

In [ ]:
scipy_corr = ndimage.correlate(small, asymmetric_kernel, mode="constant", cval=0.0)
scipy_conv = ndimage.convolve(small, asymmetric_kernel, mode="constant", cval=0.0)

print("SciPy correlation at center:", scipy_corr[2, 2])
print("SciPy convolution at center :", scipy_conv[2, 2])

assert np.isclose(manual_corr, scipy_corr[2, 2])
assert np.isclose(manual_conv, scipy_conv[2, 2])

print("\nPASS — manual calculations agree with SciPy.")

## 5. Direct Convolution Implementation


In [ ]:
# Flip the kernel explicitly so this routine implements convolution rather than correlation.
def convolve_image_manually(image, kernel):
    image = np.asarray(image, dtype=float)

    kernel = np.asarray(kernel, dtype=float)

    kh, kw = kernel.shape

    assert kh % 2 == 1 and kw % 2 == 1, \
        "Kernel dimensions must be odd."

    pad_h = kh // 2
    pad_w = kw // 2
    # Symmetric padding avoids introducing artificial zero-valued borders into the comparison.
    padded = np.pad(
        image,
        ((pad_h, pad_h), (pad_w, pad_w)),
        mode="symmetric"
    )
    kernel_flipped = np.flip(kernel, axis=(0, 1))
    output = np.zeros_like(image, dtype=float)

    for row in range(image.shape[0]):
        for col in range(image.shape[1]):

            neighbourhood = padded[
                row:row + kh,
                col:col + kw
            ]

            output[row, col] = np.sum(
                neighbourhood * kernel_flipped
            )

    return output

In [ ]:
toy_float64 = np.asarray(toy, dtype=np.float64)

kernel_float64 = np.asarray(mean_3x3, dtype=np.float64)
manual_convolution_result = convolve_image_manually(
    toy_float64,
    kernel_float64
)
scipy_convolution_result = ndimage.convolve(
    toy_float64,
    kernel_float64,
    mode="reflect"
)

maximum_convolution_difference = np.max(
    np.abs(manual_convolution_result - scipy_convolution_result)
)

print("Manual dtype:", manual_convolution_result.dtype)
print("SciPy dtype :", scipy_convolution_result.dtype)
print("Maximum absolute difference:", maximum_convolution_difference)

assert np.allclose(
    manual_convolution_result,
    scipy_convolution_result,
    rtol=1e-10,
    atol=1e-12
)

print("PASS — manual convolution matches scipy.ndimage.convolve.")

## 6. Border Handling


In [ ]:
demo = np.zeros((9, 9), dtype=np.float32)
demo[2:7, 2:7] = 200
large_mean_filter_result = np.ones((5, 5), dtype=np.float32) / 25.0
border_modes = {
    "constant": ndimage.convolve(demo, large_mean_filter_result, mode="constant", cval=0.0),
    "nearest": ndimage.convolve(demo, large_mean_filter_result, mode="nearest"),
    "reflect": ndimage.convolve(demo, large_mean_filter_result, mode="reflect"),
    "mirror": ndimage.convolve(demo, large_mean_filter_result, mode="mirror"),
    "wrap": ndimage.convolve(demo, large_mean_filter_result, mode="wrap"),
}

fig, axes = plt.subplots(2, 3, figsize=(11, 7))
axes = axes.ravel()

axes[0].imshow(demo, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")

for ax, (name, result) in zip(axes[1:], border_modes.items()):
    ax.imshow(result, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_border_modes.png", bbox_inches="tight")
plt.show()

## 7. Mean / Box Filtering


In [ ]:
mean_kernel = np.ones((3, 3), dtype=np.float32) / 9.0

mean_filtered_image = ndimage.convolve(
    einstein_gaussian_noise_image.astype(np.float32),
    mean_kernel,
    mode="reflect",
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(einstein_reference_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Reference")

axes[1].imshow(einstein_gaussian_noise_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Gaussian-noisy")

axes[2].imshow(mean_filtered_image, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("3x3 mean filter")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_mean_filter.png", bbox_inches="tight")
plt.show()

### Kernel-Size Sweep


In [ ]:
mean_sizes = [3, 5, 9]

mean_filter_results = {}

for size in mean_sizes:
    kernel = np.ones((size, size), dtype=np.float32) / (size * size)
    mean_filter_results[size] = ndimage.convolve(
        einstein_gaussian_noise_image.astype(np.float32),
        kernel,
        mode="reflect",
    )

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(einstein_gaussian_noise_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Noisy input")

for ax, size in zip(axes[1:], mean_sizes):
    ax.imshow(mean_filter_results[size], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{size}x{size} mean")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_mean_kernel_sizes.png", bbox_inches="tight")
plt.show()

## 8. Gaussian Filtering


### Gaussian-Kernel Construction


In [ ]:
# Normalize the Gaussian kernel to unit sum so smoothing preserves constant-image intensity.
def build_gaussian_kernel_2d(size: int, sigma: float) -> np.ndarray:
    assert size % 2 == 1, "Kernel size must be odd."
    assert sigma > 0, "sigma must be positive."
    radius = size // 2
    coords = np.arange(-radius, radius + 1, dtype=np.float64)
    x, y = np.meshgrid(coords, coords)

    kernel = np.exp(-(x**2 + y**2) / (2.0 * sigma**2))
    kernel /= kernel.sum()
    return kernel

g5 = build_gaussian_kernel_2d(5, 1.0)

print(np.round(g5, 4))
print("Kernel sum:", g5.sum())

In [ ]:
# Verify Gaussian separability through the equivalent 1-D construction used in the outer-product check.
def build_gaussian_kernel_1d(size: int, sigma: float) -> np.ndarray:

    radius = size // 2
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    g = np.exp(-(x**2) / (2.0 * sigma**2))
    return g / g.sum()

g1 = build_gaussian_kernel_1d(5, 1.0)
outer = np.outer(g1, g1)

print("Maximum difference between 2-D kernel and outer product:")
print(np.max(np.abs(g5 - outer)))

assert np.allclose(g5, outer)

In [ ]:
gaussian_filtered_image = ndimage.gaussian_filter(
    einstein_gaussian_noise_image.astype(np.float32),

sigma=1.5,
    mode="reflect",
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(einstein_reference_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Reference")

axes[1].imshow(einstein_gaussian_noise_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title("Gaussian-noisy")

axes[2].imshow(gaussian_filtered_image, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Gaussian filter, sigma=1.5")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_gaussian_filter.png", bbox_inches="tight")
plt.show()

In [ ]:
sigmas = [0.5, 1.0, 2.0, 3.0]

gaussian_filter_results = {
    sigma: ndimage.gaussian_filter(
        einstein_gaussian_noise_image.astype(np.float32),
        sigma=sigma,
        mode="reflect",
    )
    for sigma in sigmas
}

fig, axes = plt.subplots(1, 5, figsize=(17, 4))

axes[0].imshow(einstein_gaussian_noise_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Noisy input")

for ax, sigma in zip(axes[1:], sigmas):
    ax.imshow(gaussian_filter_results[sigma], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"sigma={sigma}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_gaussian_sigma_sweep.png", bbox_inches="tight")
plt.show()

In [ ]:
mean_5 = ndimage.uniform_filter(
    einstein_gaussian_noise_image.astype(np.float32),
    size=5,
    mode="reflect",
)
gauss_approx = ndimage.gaussian_filter(
    einstein_gaussian_noise_image.astype(np.float32),
    sigma=1.0,
    mode="reflect",
)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, img, title in zip(
    axes,
    [einstein_gaussian_noise_image, mean_5, gauss_approx],
    ["Noisy", "5x5 mean", "Gaussian sigma=1"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "06_mean_vs_gaussian.png", bbox_inches="tight")
plt.show()

## 9. Noise-Model Dependence


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, img, title in zip(
    axes,

[einstein_reference_image, einstein_gaussian_noise_image, einstein_salt_pepper_noise_image, einstein_speckle_noise_image],
    ["Reference", "Gaussian noise", "Salt-and-pepper", "Speckle noise"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "07_noise_types.png", bbox_inches="tight")
plt.show()

## 10. Median Filtering


In [ ]:
values = np.array([20, 21, 20, 22, 255, 19, 20, 21, 20])

print("Values:", values)
print("Sorted:", np.sort(values))
print("Mean  :", values.mean())
print("Median:", np.median(values))

In [ ]:
# Median filtering is evaluated on impulse noise because isolated extremes should be removed without averaging them into neighbors.
median_filtered = ndimage.median_filter(
    einstein_salt_pepper_noise_image,
    size=3,
    mode="reflect",
)
mean_on_salt = ndimage.uniform_filter(
    einstein_salt_pepper_noise_image.astype(np.float32),
    size=3,
    mode="reflect",
)

gaussian_on_salt = ndimage.gaussian_filter(
    einstein_salt_pepper_noise_image.astype(np.float32),
    sigma=1.0,
    mode="reflect",
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, img, title in zip(
    axes,
    [einstein_salt_pepper_noise_image, mean_on_salt, gaussian_on_salt, median_filtered],
    ["Salt-and-pepper", "Mean", "Gaussian", "Median"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "08_median_filter.png", bbox_inches="tight")
plt.show()

### Median-Window Sweep


In [ ]:
median_sizes = [3, 5, 7]

median_results = {
    size: ndimage.median_filter(einstein_salt_pepper_noise_image, size=size, mode="reflect")
    for size in median_sizes
}

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(einstein_salt_pepper_noise_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Noisy")

for ax, size in zip(axes[1:], median_sizes):
    ax.imshow(median_results[size], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{size}x{size} median")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "09_median_sizes.png", bbox_inches="tight")
plt.show()

## 11. Bilateral Filtering


In [ ]:
# Bilateral filtering couples spatial and intensity similarity to denoise while retaining edge transitions.
bilateral = cv2.bilateralFilter(
    einstein_gaussian_noise_image,

d=9,

    sigmaColor=45,
    sigmaSpace=45,
)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, img, title in zip(
    axes,
    [einstein_reference_image, einstein_gaussian_noise_image, gaussian_filtered_image, bilateral],
    ["Reference", "Noisy", "Gaussian filter", "Bilateral filter"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "10_bilateral_filter.png", bbox_inches="tight")
plt.show()

### Bilateral-Parameter Sweep


In [ ]:
sigma_colors = [10, 30, 60, 120]
bilateral_results = {
    sc: cv2.bilateralFilter(
        einstein_gaussian_noise_image,
        d=9,
        sigmaColor=sc,
        sigmaSpace=45,
    )
    for sc in sigma_colors
}

fig, axes = plt.subplots(1, 5, figsize=(17, 4))

axes[0].imshow(einstein_gaussian_noise_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Noisy")

for ax, sc in zip(axes[1:], sigma_colors):
    ax.imshow(bilateral_results[sc], cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"sigmaColor={sc}")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "11_bilateral_parameter_sweep.png", bbox_inches="tight")
plt.show()

## 12. Quantitative Denoising Metrics


In [ ]:
def mean_absolute_error(reference, test):
    ref = reference.astype(np.float64)
    tst = np.asarray(test, dtype=np.float64)
    return float(np.mean(np.abs(ref - tst)))

def mean_squared_error(reference, test):
    ref = reference.astype(np.float64)
    tst = np.asarray(test, dtype=np.float64)
    return float(np.mean((ref - tst) ** 2))

def root_mean_squared_error(reference, test):
    return math.sqrt(mean_squared_error(reference, test))

def peak_signal_to_noise_ratio(reference, test, peak=255.0):
    value = mean_squared_error(reference, test)

    if value == 0:
        return float("inf")
    return 10.0 * math.log10((peak ** 2) / value)

def build_metric_summary(name, reference, test):
    return (
        name,
        mean_absolute_error(reference, test),
        mean_squared_error(reference, test),
        root_mean_squared_error(reference, test),
        peak_signal_to_noise_ratio(reference, test),
    )

### Gaussian-Noise Comparison


In [ ]:
gaussian_candidates = [
    build_metric_summary("Noisy", einstein_reference_image, einstein_gaussian_noise_image),
    build_metric_summary("Mean 3x3", einstein_reference_image, mean_filtered_image),
    build_metric_summary("Gaussian sigma=1.5", einstein_reference_image, gaussian_filtered_image),
    build_metric_summary("Bilateral", einstein_reference_image, bilateral),
]

print(f"{'Method':24s} {'MAE':>10s} {'MSE':>12s} {'RMSE':>10s} {'PSNR(dB)':>10s}")
print("-" * 72)

for name, a, m, r, p in gaussian_candidates:
    print(f"{name:24s} {a:10.3f} {m:12.3f} {r:10.3f} {p:10.3f}")

### Impulse-Noise Comparison


In [ ]:
salt_candidates = [
    build_metric_summary("Noisy", einstein_reference_image, einstein_salt_pepper_noise_image),
    build_metric_summary("Mean 3x3", einstein_reference_image, mean_on_salt),
    build_metric_summary("Gaussian sigma=1", einstein_reference_image, gaussian_on_salt),
    build_metric_summary("Median 3x3", einstein_reference_image, median_filtered),
]

print(f"{'Method':24s} {'MAE':>10s} {'MSE':>12s} {'RMSE':>10s} {'PSNR(dB)':>10s}")
print("-" * 72)

for name, a, m, r, p in salt_candidates:
    print(f"{name:24s} {a:10.3f} {m:12.3f} {r:10.3f} {p:10.3f}")

## 13. Edge Preservation as a Secondary Check


In [ ]:
# Use gradient magnitude as an edge-retention diagnostic so denoising is not judged by pixel error alone.
def compute_sobel_gradient_magnitude(image):
    arr = np.asarray(image, dtype=np.float32)
    gx = ndimage.sobel(arr, axis=1, mode="reflect")
    gy = ndimage.sobel(arr, axis=0, mode="reflect")
    return np.hypot(gx, gy)

edge_ref = compute_sobel_gradient_magnitude(einstein_reference_image)

edge_mean = compute_sobel_gradient_magnitude(mean_filtered_image)
edge_gauss = compute_sobel_gradient_magnitude(gaussian_filtered_image)
edge_bilat = compute_sobel_gradient_magnitude(bilateral)

print("Mean gradient magnitude")
print(f"Reference : {edge_ref.mean():.3f}")
print(f"Mean      : {edge_mean.mean():.3f}")
print(f"Gaussian  : {edge_gauss.mean():.3f}")
print(f"Bilateral : {edge_bilat.mean():.3f}")

## 14. Sharpening with the Laplacian


In [ ]:
laplacian_kernel = np.array(
    [
        [0, 1, 0],
        [1, -4, 1],
        [0, 1, 0],
    ],
    dtype=np.float32,
)

print(laplacian_kernel)
print("Kernel sum:", laplacian_kernel.sum())

In [ ]:
moon_float = moon.astype(np.float32)
laplacian = ndimage.convolve(
    moon_float,
    laplacian_kernel,
    mode="reflect",
)

lap_sharp = np.clip(
    moon_float - laplacian,
    0,
    255,
).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(moon, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Blurred moon")

axes[1].imshow(laplacian, cmap="gray")
axes[1].set_title("Laplacian response")

axes[2].imshow(lap_sharp, cmap="gray", vmin=0, vmax=255)
axes[2].set_title("Laplacian sharpened")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "12_laplacian_sharpening.png", bbox_inches="tight")
plt.show()

## 15. Unsharp Masking and High-Boost Filtering


In [ ]:
blurred = ndimage.gaussian_filter(moon_float, sigma=2.0, mode="reflect")
detail_mask = moon_float - blurred

unsharp = np.clip(
    moon_float + 1.0 * detail_mask,
    0,
    255,
).astype(np.uint8)
# Reuse the extracted high-frequency detail with a larger gain to contrast unsharp and high-boost sharpening.
high_boost = np.clip(
    moon_float + 1.8 * detail_mask,
    0,
    255,
).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

for ax, img, title in zip(
    axes,
    [moon, blurred, unsharp, high_boost],
    ["Input", "Gaussian blur", "Unsharp k=1", "High-boost k=1.8"],
):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "13_unsharp_highboost.png", bbox_inches="tight")
plt.show()

## 16. First Derivatives and Image Gradients


In [ ]:
gx = ndimage.sobel(ascent.astype(np.float32), axis=1, mode="reflect")
gy = ndimage.sobel(ascent.astype(np.float32), axis=0, mode="reflect")
magnitude = np.hypot(gx, gy)
orientation = np.arctan2(gy, gx)

fig, axes = plt.subplots(1, 4, figsize=(15, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")

axes[1].imshow(np.abs(gx), cmap="gray")
axes[1].set_title("|Gx|")

axes[2].imshow(np.abs(gy), cmap="gray")
axes[2].set_title("|Gy|")

axes[3].imshow(magnitude, cmap="gray")
axes[3].set_title("Gradient magnitude")

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "14_sobel_gradients.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(orientation, cmap="twilight", vmin=-np.pi, vmax=np.pi)
ax.set_title("Gradient orientation (radians)")
ax.axis("off")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "15_gradient_orientation.png", bbox_inches="tight")
plt.show()

## 17. Sobel vs Prewitt vs Scharr


In [ ]:
ascent_float_image = ascent.astype(np.float32)
prewitt_x = ndimage.prewitt(ascent_float_image, axis=1, mode="reflect")
prewitt_y = ndimage.prewitt(ascent_float_image, axis=0, mode="reflect")
prewitt_mag = np.hypot(prewitt_x, prewitt_y)

sobel_x = ndimage.sobel(ascent_float_image, axis=1, mode="reflect")
sobel_y = ndimage.sobel(ascent_float_image, axis=0, mode="reflect")
sobel_mag = np.hypot(sobel_x, sobel_y)

scharr_x = cv2.Scharr(ascent_float_image, cv2.CV_32F, 1, 0)
scharr_y = cv2.Scharr(ascent_float_image, cv2.CV_32F, 0, 1)
scharr_mag = np.hypot(scharr_x, scharr_y)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, img, title in zip(
    axes,
    [prewitt_mag, sobel_mag, scharr_mag],
    ["Prewitt", "Sobel", "Scharr"],
):
    ax.imshow(img, cmap="gray")
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "16_derivative_operators.png", bbox_inches="tight")
plt.show()

## 18. Border Effects on a Real Image


In [ ]:
kernel_15 = np.ones((15, 15), dtype=np.float32) / (15 * 15)
ascent_float_image = ascent.astype(np.float32)
padding_results = {
    "constant": ndimage.convolve(ascent_float_image, kernel_15, mode="constant", cval=0.0),
    "nearest": ndimage.convolve(ascent_float_image, kernel_15, mode="nearest"),
    "reflect": ndimage.convolve(ascent_float_image, kernel_15, mode="reflect"),
    "wrap": ndimage.convolve(ascent_float_image, kernel_15, mode="wrap"),
}

fig, axes = plt.subplots(1, 5, figsize=(17, 4))

axes[0].imshow(ascent, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")

for ax, (name, img) in zip(axes[1:], padding_results.items()):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)

for ax in axes:
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "17_padding_real_image.png", bbox_inches="tight")
plt.show()

## 19. Filtering RGB Images


In [ ]:
taj_gaussian = ndimage.gaussian_filter(
    taj_reference_image.astype(np.float32),

sigma=(1.5, 1.5, 0),
    mode="reflect",
)

taj_# Bilateral filtering couples spatial and intensity similarity to denoise while retaining edge transitions.
bilateral = cv2.bilateralFilter(
    taj_reference_image,
    d=9,
    sigmaColor=45,
    sigmaSpace=45,
)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, img, title in zip(
    axes,
    [taj_reference_image, np.clip(taj_gaussian, 0, 255).astype(np.uint8), taj_bilateral],
    ["RGB reference", "Gaussian RGB", "Bilateral RGB"],
):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

fig.tight_layout()
fig.savefig(OUTPUT_DIR / "18_color_filtering.png", bbox_inches="tight")
plt.show()

## 20. Numerical Safety


In [ ]:
example_uint8 = np.array([0, 10, 250, 255], dtype=np.uint8)
unsafe = example_uint8 + np.uint8(20)
safe = example_uint8.astype(np.float32) + 20

print("Original:", example_uint8)
print("Unsafe uint8 + 20:", unsafe)
print("Safe float + 20 :", safe)
print("Clipped result  :", np.clip(safe, 0, 255).astype(np.uint8))

## 21. Filter Selection Criteria

In [ ]:
best_gaussian_noise_method = min(
    gaussian_candidates[1:],
    key=lambda row: row[2],
)

best_impulse_noise_method = min(
    salt_candidates[1:],
    key=lambda row: row[2],
)
edge_retention = {
    "Mean 3x3": float(edge_mean.mean() / max(edge_ref.mean(), 1e-12)),
    "Gaussian sigma=1.5": float(edge_gauss.mean() / max(edge_ref.mean(), 1e-12)),
    "Bilateral": float(edge_bilat.mean() / max(edge_ref.mean(), 1e-12)),
}

print("Best Gaussian-noise restoration by MSE:")
print(
    f"  {best_gaussian_noise_method[0]} "
    f"(MSE={best_gaussian_noise_method[2]:.3f}, "
    f"PSNR={best_gaussian_noise_method[4]:.3f} dB)"
)

print("\nBest impulse-noise restoration by MSE:")
print(
    f"  {best_impulse_noise_method[0]} "
    f"(MSE={best_impulse_noise_method[2]:.3f}, "
    f"PSNR={best_impulse_noise_method[4]:.3f} dB)"
)

print("\nRelative mean-gradient retention:")
for method, ratio in edge_retention.items():
    print(f"  {method:20s}: {ratio:.3f}")

## 22. Standard Spatial-Filtering Workflow

In [ ]:
def apply_spatial_filter(
    image,
    method="gaussian",
    kernel_size=5,
    sigma=1.5,
):
    image = np.asarray(image, dtype=np.float32)
    if image.ndim != 2:
        raise ValueError("apply_spatial_filter expects a 2-D grayscale image.")
    if kernel_size <= 0 or kernel_size % 2 == 0:
        raise ValueError("kernel_size must be a positive odd integer.")

    if method == "mean":
        return ndimage.uniform_filter(
            image,
            size=kernel_size,
            mode="reflect",
        )

    if method == "gaussian":
        if sigma <= 0:
            raise ValueError("sigma must be strictly positive.")
        return ndimage.gaussian_filter(
            image,
            sigma=sigma,
            mode="reflect",
        )

    if method == "median":
        return ndimage.median_filter(
            image,
            size=kernel_size,
            mode="reflect",
        )

    raise ValueError(
        "method must be one of: 'mean', 'gaussian', 'median'."
    )
workflow_result = apply_spatial_filter(
    einstein_gaussian_noise_image,
    method="gaussian",
    kernel_size=5,
    sigma=1.5,
)

assert workflow_result.shape == einstein_gaussian_noise_image.shape
assert np.all(np.isfinite(workflow_result))

print(
    "Workflow example:",
    "Gaussian sigma=1.5",
    f"shape={workflow_result.shape}",
    f"MSE={mean_squared_error(einstein_reference_image, workflow_result):.3f}",
)

## 23. Validation Checks


In [ ]:
assert mean_filtered_image.shape == einstein_reference_image.shape
assert gaussian_filtered_image.shape == einstein_reference_image.shape
assert median_filtered.shape == einstein_reference_image.shape
assert bilateral.shape == einstein_reference_image.shape
assert np.isclose(mean_kernel.sum(), 1.0)

assert np.isclose(g5.sum(), 1.0)
assert np.isclose(sobel_x_kernel.sum(), 0.0)
assert np.isclose(laplacian_kernel.sum(), 0.0)
assert np.allclose(manual_convolution_result, scipy_convolution_result)

assert np.all(magnitude >= 0)

assert taj_gaussian.shape == taj_reference_image.shape
assert taj_bilateral.shape == taj_reference_image.shape

assert lap_sharp.min() >= 0 and lap_sharp.max() <= 255
assert unsharp.min() >= 0 and unsharp.max() <= 255

print("All spatial-filtering validation checks passed.")
REQUIRED_OUTPUTS = [
    "01_border_modes.png",
    "02_mean_filter.png",
    "03_mean_kernel_sizes.png",
    "04_gaussian_filter.png",
    "05_gaussian_sigma_sweep.png",
    "06_mean_vs_gaussian.png",
    "07_noise_types.png",
    "08_median_filter.png",
    "09_median_sizes.png",
    "10_bilateral_filter.png",
    "11_bilateral_parameter_sweep.png",
    "12_laplacian_sharpening.png",
    "13_unsharp_highboost.png",
    "14_sobel_gradients.png",
    "15_gradient_orientation.png",
    "16_derivative_operators.png",
    "17_padding_real_image.png",
    "18_color_filtering.png",
]
missing_outputs = [
    output_name
    for output_name in REQUIRED_OUTPUTS
    if not (OUTPUT_DIR / output_name).exists()
]
if missing_outputs:
    raise FileNotFoundError(
        "Missing outputs: " + ", ".join(missing_outputs)
    )

print(f"Output validation passed: {len(REQUIRED_OUTPUTS)} files.")
